## תוצרי ההתאמה: שגיאות המקדמים ושאריות

`linear_fit` שכתבנו בסעיף הקודם מחזירה `m, b` -- אבל שני מספרים בלי טווח אמון לא אומרים הרבה. השאלה החשובה: **כמה בטוחים אנחנו** ב-`m` וב-`b`? לשם כך צריך שני מרכיבים:

1. **שאריות** (residuals): `resid = y - (m*x + b)` -- ההפרש בין המדידה לתחזית המודל, לכל נקודה.
2. **שגיאות המקדמים** (standard errors): מחושבות מהפיזור של השאריות, ומגלמות "כמה רחוקה ההתאמה מלהיות מושלמת".

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())
x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])

def linear_fit(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

m, b = linear_fit(x, y)
y_pred = m*x + b
resid = y - y_pred
print("שאריות:", np.round(resid, 3))
print("ממוצע השאריות:", resid.mean())

שימו לב: ממוצע השאריות **תמיד** קרוב ל-0 (עד שגיאת עיגול), בכל התאמת ריבועים פחותים עם חיתוך חופשי -- זו תוצאה מתמטית ישירה של הנוסחה, לא משהו שצריך "לבדוק שמתקיים במקרה שלנו".

### שגיאות המקדמים

עם `n` נקודות ו-2 פרמטרים (`m`, `b`) שהותאמו, נשארות `n-2` **דרגות חופש**:

$$s_y = \sqrt{\frac{\sum \text{resid}_i^2}{n-2}} \qquad SE_m = \frac{s_y}{\sqrt{\sum (x_i-\bar x)^2}} \qquad SE_b = s_y\sqrt{\frac{\sum x_i^2}{n\sum (x_i-\bar x)^2}}$$

`n-2` (ולא `n`) כי כל פרמטר שמותאם לנתונים "צורך" דרגת חופש אחת -- אותו עיקרון בדיוק כמו `ddof=1` בסטיית תקן של דגימה (סעיף 11.2), הפעם עם שני פרמטרים.

In [ ]:
n = len(x)
dof = n - 2
s_y = np.sqrt(np.sum(resid**2) / dof)

x_bar = x.mean()
SE_m = s_y / np.sqrt(np.sum((x - x_bar)**2))
SE_b = s_y * np.sqrt(np.sum(x**2) / (n * np.sum((x - x_bar)**2)))

print(f"m = {m:.3f} +/- {SE_m:.3f}")
print(f"b = {b:.3f} +/- {SE_b:.3f}")

### באג נפוץ: דרגות חופש שגויות

לחלק ב-`n` במקום ב-`n-2` (לשכוח ש-2 פרמטרים כבר הותאמו לנתונים) -- לא זורק שגיאה, אבל **מקטין באופן שיטתי** את `s_y` ולכן גם את `SE_m`, `SE_b` -- כלומר, נותן תחושת ביטחון גבוהה יותר ממה שמוצדק.

In [ ]:
s_y_wrong = np.sqrt(np.sum(resid**2) / n)   # n, לא n-2
SE_m_wrong = s_y_wrong / np.sqrt(np.sum((x - x_bar)**2))
print(f"SE_m נכון (n-2): {SE_m:.4f}")
print(f"SE_m שגוי (n):   {SE_m_wrong:.4f}  - קטן יותר, נראה 'טוב מדי'")

### נסו בעצמכם

חשבו את `resid`, `s_y`, `SE_m`, `SE_b` עבור התאמה של `run_id` מול `v0_measured` בתוך זווית 60 בלבד.

In [ ]:
# sub60 = df_clean[df_clean["angle_deg"] == 60]
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
sub60 = df_clean[df_clean["angle_deg"] == 60]
x60 = sub60["run_id"].to_numpy(dtype=float)
y60 = sub60["v0_measured"].to_numpy(dtype=float)

m60, b60 = linear_fit(x60, y60)
resid60 = y60 - (m60*x60 + b60)
n60 = len(x60)
dof60 = n60 - 2
s_y60 = np.sqrt(np.sum(resid60**2) / dof60)
x_bar60 = x60.mean()
SE_m60 = s_y60 / np.sqrt(np.sum((x60 - x_bar60)**2))
SE_b60 = s_y60 * np.sqrt(np.sum(x60**2) / (n60 * np.sum((x60 - x_bar60)**2)))

print(f"m = {m60:.4f} +/- {SE_m60:.4f}")
print(f"b = {b60:.3f} +/- {SE_b60:.3f}")
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "למה מחלקים בסכום ריבועי השאריות ב-(n-2) ולא ב-n בחישוב s_y?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי שני פרמטרים (m ו-b) כבר הותאמו לנתונים, ו'צרכו' שתי דרגות חופש", "correct": True, "feedback": "נכון - אותו עיקרון כמו ddof=1, מוכפל."},
            {"answer": "זו רק קונבנציה שרירותית ללא סיבה מתמטית", "correct": False, "feedback": "לא - יש לה בסיס בתורת האומדים הלא-מוטים."},
            {"answer": "כדי שהתוצאה תמיד תהיה חיובית", "correct": False, "feedback": "סכום ריבועים תמיד חיובי בלי קשר למכנה."},
            {"answer": "כדי לתקן שגיאת עיגול בחישוב", "correct": False, "feedback": "לא קשור לעיגול — יש לזה בסיס סטטיסטי מדויק (אומד לא-מוטה), לא תיקון טכני."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

ארזו את כל החישוב (fit + resid + SE) לפונקציה אחת `fit_with_errors(x, y)` שמחזירה מילון עם `m`, `b`, `SE_m`, `SE_b`, `resid`. הריצו אותה על נתוני ה-`x, y` הראשיים (הטווח מול sin(2*theta)) ועל זווית 60 בנפרד, והדפיסו את שתיהן.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
def fit_with_errors(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m, b = linear_fit(x, y)
    y_pred = m*x + b
    resid = y - y_pred
    n = len(x)
    dof = n - 2
    s_y = np.sqrt(np.sum(resid**2) / dof)
    x_bar = x.mean()
    SE_m = s_y / np.sqrt(np.sum((x - x_bar)**2))
    SE_b = s_y * np.sqrt(np.sum(x**2) / (n * np.sum((x - x_bar)**2)))
    return {"m": m, "b": b, "SE_m": SE_m, "SE_b": SE_b, "resid": resid}

result_main = fit_with_errors(x, y)
result_60 = fit_with_errors(sub60["run_id"], sub60["v0_measured"])
print(result_main["m"], result_main["SE_m"])
print(result_60["m"], result_60["SE_m"])
```
`````